### Ingestion del archivo "production_country.json"

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

#### Paso 1 - Leer el archivo JSON usando "DataFrameReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
prod_county_schema = StructType(fields=[
    StructField("movieId", IntegerType(), True),
    StructField("countryId", IntegerType(), True)
    ])

In [0]:
prod_county_df = spark.read \
            .schema(prod_county_schema) \
            .option("multiLine", "true") \
            .json(f"{bronze_folder_path}/{v_file_date}/production_country")

# production_country_*.csv

In [0]:
display(prod_county_df)

In [0]:
prod_county_df.count()   # ver la cantidad de registros que se han cargado 

#### Paso 2 - renombrar columnas y agregar columnas nuevas
1. renombrar "movieId" a "movie_id"
2. renombrar "countryId" a "country_id"
3. Agregar las columnas "ingestion_date" y "environment"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
prod_country_withcolumns_df = add_ingestion_date(prod_county_df) \
                            .withColumnsRenamed({"movieId": "movie_id", "countryId": "country_id"}) \
                            .withColumn("environment", lit(v_environment)) \
                            .withColumn("file_date", lit(v_file_date))

In [0]:
display(prod_country_withcolumns_df)

#### Paso 3 - Escribir salida en un formato "Parquet"

In [0]:
# prod_country_withcolumns_df.write.mode("overwrite").parquet(f"{silver_folder_path}/productions_countries")

In [0]:
# display(spark.read.parquet("abfss://silver@moviehistory22.dfs.core.windows.net/productions_countries"))

In [0]:
# overwrite_partition("movie_silver", "productions_countries","file_date",v_file_date)

In [0]:
# prod_country_withcolumns_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.productions_countries")
# prod_country_withcolumns_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.productions_countries")

In [0]:
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.file_date=src.file_date'
merge_delta_lake(prod_country_withcolumns_df, "movie_silver", "productions_countries", merge_condition, "file_date")

In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.productions_countries
GROUP BY file_date

In [0]:
%sql
SELECT * FROM movie_silver.productions_countries

In [0]:
dbutils.notebook.exit("Success")